# Processing API Payloads

## What you'll practice

API JSON is never the shape you want: camelCase keys, envelopes, timestamps in three formats, duplicates across pages, nulls. These exercises turn raw
payloads (including `datasets/json/api/orders.json`) into clean, typed records — and reconcile them against a local copy.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/apis/test_07_02_json_api_processing.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/07_apis/02_json_api_processing.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))                                # make `labkit` importable from the notebook
from labkit.mock_api import MockSession, BASE_URL
import requests

session = MockSession()                                     # offline fake with the same surface as requests.Session
resp = session.get(f"{BASE_URL}/customers", params={"page": 1, "page_size": 3}, timeout=5)
print(resp.status_code, resp.json()["total"], [c["customer_id"] for c in resp.json()["data"]])
# Prefer real HTTP? In a terminal:  python -m labkit.mock_server --port 8765
# then use requests.Session() with base_url "http://127.0.0.1:8765/v1" — your functions shouldn't change at all.

## Exercise 1 — 🟢 Easy · Unwrap response envelopes

**Implement:** `extract_items(payload: dict, path: str = 'data') -> list`

Return the list at dotted `path` (`"result.items"`). Missing keys or `None` → `[]`. A non-list value at the path → `TypeError`.

**Input:** `payload: dict, path: str = 'data'`

**Expected output:** `list`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example**

```python
extract_items({'data': [1]})
# -> [1]
```

**Check:** `pytest tests/apis/test_07_02_json_api_processing.py -k test_extract_items`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · camelCase → snake_case, recursively

**Implement:** `to_snake(name: str) -> str`, `snake_keys(obj)`

`to_snake("customerId")` → `customer_id`; handle acronyms: `"HTTPStatusCode"` → `http_status_code`, `"orderID"` → `order_id`, `"already_snake"` unchanged, `"v2Name"` → `v2_name`.
`snake_keys` converts dict keys at every depth, including dicts inside lists; values untouched.

**Input:** `name: str`

**Expected output:** `str`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example**

```python
to_snake('customerId')
# -> 'customer_id'
```

**Check:** `pytest tests/apis/test_07_02_json_api_processing.py -k "test_to_snake or test_snake_keys"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Parse timestamps in every API flavour

**Implement:** `parse_ts(value) -> datetime`

Return a timezone-aware UTC `datetime` from: ISO strings with `Z` or an offset (`2026-01-05T10:00:00+05:30`), naive ISO strings (assume UTC), epoch seconds (int/float),
epoch **milliseconds** (numbers > 10¹¹), and numeric strings of either. Anything else → `ValueError`.

**Input:** `value`

**Expected output:** `datetime`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_parse_ts, test_parse_ts_invalid` in `tests/apis/test_07_02_json_api_processing.py`.

**Check:** `pytest tests/apis/test_07_02_json_api_processing.py -k "test_parse_ts or test_parse_ts_invalid"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Normalize and deduplicate API orders

**Implement:** `normalize_orders(raw: list[dict]) -> list[dict]`

Raw orders from the mock API (`datasets/json/api/orders.json`) contain duplicates (same `order_id`, possibly different `updated_at`) and some `unit_price = None`.
Produce typed records `{order_id, customer_id, product_id, quantity: int, unit_price: float|None, amount: float|None (rounded 2), status: UPPER, order_ts: datetime UTC, updated_at: datetime UTC}`,
keeping only the version with the **latest** `updated_at` per `order_id` (ties → the later one in input). Return sorted by `order_id`.

**Input:** `raw: list[dict]`

**Expected output:** `list[dict]`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_normalize_orders, test_normalize_orders_real_api_data` in `tests/apis/test_07_02_json_api_processing.py`.

**Check:** `pytest tests/apis/test_07_02_json_api_processing.py -k "test_normalize_orders or test_normalize_orders_real_api_data"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Reconcile API data against the warehouse · 🧩 Mini-project

**Implement:** `reconcile(source: list[dict], target: list[dict], key: str, fields: list[str], tolerance: float = 0.0) -> dict`

After loading, data engineers prove the load is right. Compare `source` (API) with `target` (warehouse) by `key`:
`{"source_count", "target_count", "matched", "missing_in_target": [keys], "unexpected_in_target": [keys], "mismatches": [{"key", "field", "source", "target"}], "duplicate_keys": {"source": [...], "target": [...]}}`.
Numeric fields (int/float on both sides) match if `abs(a - b) <= tolerance`; strings are compared after `strip()`; `None` only equals `None`. All key lists sorted;
mismatches sorted by (key, field order). `matched` counts keys present on both sides with no mismatching field. With duplicate keys, compare the **last** occurrence.

**Input:** `source: list[dict], target: list[dict], key: str, fields: list[str], tolerance: float = 0.0`

**Expected output:** `dict`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_reconcile, test_reconcile_none_and_empty` in `tests/apis/test_07_02_json_api_processing.py`.

**Check:** `pytest tests/apis/test_07_02_json_api_processing.py -k "test_reconcile or test_reconcile_none_and_empty"`

**YOUR CODE ↓**